# Phase 6 — 계측 핵심 알고리즘 (예제)

라인 프로파일에서 엣지를 찾아 두께를 재고, 여러 위치에서 반복 측정해 통계를 냅니다. 각 셀을 순서대로 실행하십시오.

## 0. 그래프 한글 폰트 설정

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _n in ['Malgun Gothic','NanumGothic','AppleGothic','Noto Sans CJK KR','Noto Sans KR']:
    if _n in {f.name for f in fm.fontManager.ttflist}:
        plt.rcParams['font.family'] = _n; break
plt.rcParams['axes.unicode_minus'] = False

## 1. 예제 이미지와 헬퍼 함수
밝은 층(참 두께 35px)이 있는 예제 이미지를 만들고, 프로파일·엣지 헬퍼를 정의합니다. 실제 파일이 있으면 `img = cv2.imread('경로', cv2.IMREAD_GRAYSCALE)` 로 바꾸면 됩니다.

In [ ]:
import numpy as np, cv2
np.random.seed(3)

H,W=220,360; bg,fg=90,185
img=np.full((H,W),bg,float); img[95:130,:]=fg   # 참 두께 = 35 px
for x in range(W):
    img[:,x]=np.roll(img[:,x], int(3*np.sin(x/50.0)))
img=np.clip(img+np.random.normal(0,8,(H,W)),0,255).astype(np.uint8)

scale = 0.25   # nm/pixel (Phase 4 에서 파일로부터 읽은 값)

def profile_at(im, x):
    col=im[:,x].astype(float)
    return cv2.GaussianBlur(col.reshape(-1,1),(1,7),0).ravel()

def crossings(y, level):
    """밝기 곡선 y 가 level 을 지나는 위치들(서브픽셀)."""
    xs=[]
    for i in range(len(y)-1):
        if (y[i]-level)*(y[i+1]-level) < 0:
            t=(level-y[i])/(y[i+1]-y[i]); xs.append(i+t)
    return xs
print('준비 완료:', img.shape)

## 2. 라인 프로파일 추출
측정 방향으로 한 줄의 밝기를 뽑습니다. 층을 가로지르면 밝기가 급변합니다.

In [ ]:
x0=180
prof=profile_at(img, x0)

plt.figure(figsize=(5,3))
plt.plot(prof, color='#1F2A37')
plt.xlabel('row (pixel)'); plt.ylabel('intensity'); plt.title('line profile')
plt.tight_layout(); plt.show()

## 3. 엣지 위치 (50% 크로싱, 서브픽셀)
배경·층의 중간값(50% 레벨)을 지나는 위치가 경계입니다. `crossings` 는 선형 보간으로 서브픽셀 위치를 줍니다.

In [ ]:
level=(prof.min()+prof.max())/2     # 50% 레벨
cr=crossings(prof, level)
e1,e2=cr[0],cr[-1]
print('50% 레벨 :', round(level,1))
print('엣지 위치 :', round(e1,2), round(e2,2))

## 3-1. 인접 2픽셀 선형 보간 (가장 기본)
임계값(level)을 사이에 둔 **인접 두 픽셀** i(값 y0), i+1(값 y1) 사이를 직선으로 보간해 실수 엣지 위치를 구합니다.
`t = (level - y0) / (y1 - y0)` 이면 `edge = i + t` 입니다.

In [ ]:
def edge_two_pixel(prof, level):
    """level 을 처음으로 상향 통과하는 인접 두 픽셀 사이를 선형 보간."""
    for i in range(len(prof)-1):
        y0, y1 = prof[i], prof[i+1]
        if y0 < level <= y1:            # level 을 사이에 둔 두 픽셀
            t = (level - y0) / (y1 - y0)
            return i + t
    return None

# 예시: 픽셀 100=120, 101=168, level=135
t = (135 - 120) / (168 - 120)
print('t =', round(t,3), '-> edge = 100 +', round(t,3), '=', round(100+t,3))   # 100.312

실제 프로파일의 상승 엣지에 적용해 봅니다. (내려가는 엣지는 조건을 반대로 두면 됩니다.)

In [ ]:
lv = (prof.min() + prof.max()) / 2
rise = edge_two_pixel(prof, lv)          # 올라가는 첫 엣지
print('상승 엣지(2픽셀 보간) :', round(rise, 2))

## 3-2. gradient 정점 서브픽셀 (포물선 피팅)
엣지는 밝기의 1차 미분(gradient)이 최대인 곳으로도 찾습니다. 그 정점은 2차 미분=0 인 지점이며, 픽셀은 정수뿐이라 정점이 픽셀 사이에 오면 **정수 정점 주변 3점에 포물선을 피팅**해 실수 위치를 구합니다.

In [ ]:
def peak_subpixel(prof):
    g = np.abs(np.gradient(prof))
    i = int(np.argmax(g))                 # 정수 정점
    gm, g0, gp = g[i-1], g[i], g[i+1]
    d = 0.5*(gm - gp) / (gm - 2*g0 + gp)  # 꼭짓점 오프셋 (-0.5~0.5)
    return i + d

# 실제 엣지가 100.7 인 완만한 계단으로 검증
rr = np.arange(80, 121)
edge = 90 + 95/(1 + np.exp(-(rr - 100.7)*1.1))
i_int = 80 + int(np.argmax(np.abs(np.gradient(edge))))
print('정수 정점 :', i_int)                 # 101
print('서브픽셀  :', round(80 + peak_subpixel(edge), 3))   # 약 100.72

# 정수(101)보다 참값(100.7)에 가까운 실수 위치를 얻음

## 4. 두께 측정 (px → nm)
두 엣지 간 거리가 픽셀 두께, scale 을 곱하면 실제 두께입니다.

In [ ]:
thick_px=e2-e1
thick_nm=thick_px*scale
print(f'두께 = {thick_px:.2f} px = {thick_nm:.2f} nm')   # 약 35 px

## 5. 다중 위치 반복 측정 (for · continue · 통계)
한 곳만 재면 불안정합니다. 여러 위치에서 반복 측정하되, 실패한 위치는 `continue` 로 건너뜁니다.

In [ ]:
def measure_thickness(im, x):
    y=profile_at(im, x)
    lv=(y.min()+y.max())/2
    cr=crossings(y, lv)
    if len(cr) < 2:
        return None          # 측정 실패
    return cr[-1]-cr[0]

results=[]
for x in range(30, 340, 15):
    t=measure_thickness(img, x)
    if t is None:
        continue             # 실패는 건너뜀
    results.append(t)

results=np.array(results)
print('측정 개수 :', len(results))
print(f'평균 두께 = {results.mean()*scale:.2f} nm  (± {results.std()*scale:.2f})')

## 6. 면적 측정
마스크의 픽셀 수 × scale² 로 면적을 구합니다. (윤곽선은 cv2.contourArea)

In [ ]:
mask=cv2.inRange(img, 140, 255)          # 밝은 층 선택
area_px=cv2.countNonZero(mask)
area_nm2=area_px*scale**2
print(f'면적 = {area_px} px = {area_nm2:.1f} nm^2')

## 6-1. contourArea 는 어떻게 계산되나 (신발끈 공식)
윤곽선(contour)은 영역의 외곽 경계 픽셀을 순서대로 이은 **점들의 목록**입니다. `contourArea` 는 이 점들로 **신발끈(shoelace) 공식**을 계산해 다각형 넓이(픽셀²)를 구합니다. 픽셀을 세는 `countNonZero` 와는 경계 처리 방식이 달라 값이 약간 다릅니다.

In [ ]:
poly = np.zeros((160,220), np.uint8)
cv2.fillPoly(poly, [np.array([[40,30],[170,45],[180,120],[60,130]])], 255)
cnts, _ = cv2.findContours(poly, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print('윤곽선 점 개수      :', len(cnts[0]))
print('contourArea (신발끈):', cv2.contourArea(cnts[0]))
print('countNonZero (픽셀) :', cv2.countNonZero(poly))

# 두 값은 가깝지만 동일하지 않음 (경계 픽셀 포함/제외 차이)

**신발끈(shoelace) 공식을 직접 계산해 보기**
꼭짓점 (x, y) 를 순서대로 넣어 `A = ½·|Σ(xᵢ·yᵢ₊₁ − xᵢ₊₁·yᵢ)|` 을 계산하면 cv2.contourArea 와 (거의) 같습니다.

In [ ]:
def shoelace(cnt):
    p = cnt.reshape(-1, 2).astype(float)
    x, y = p[:,0], p[:,1]
    # np.roll 로 (i+1) 항을 만든다 (마지막 -> 처음으로 순환)
    return 0.5*abs(np.dot(x, np.roll(y,-1)) - np.dot(y, np.roll(x,-1)))

print('직접 계산(신발끈) :', shoelace(cnts[0]))
print('cv2.contourArea  :', cv2.contourArea(cnts[0]))
# 두 값이 일치 -> contourArea 는 신발끈 공식으로 다각형 넓이를 구하는 것

## 7. 템플릿 매칭 (위치 자동 탐색)
반복 구조/마커를 견본으로 찾습니다. 측정 위치를 자동으로 잡을 때 씁니다.

In [ ]:
tm=np.full((160,360),70,np.uint8)
for cx in [60,140,220,300]:
    cv2.circle(tm,(cx,80),16,210,-1)
tm=np.clip(tm+np.random.normal(0,6,tm.shape),0,255).astype(np.uint8)

templ=tm[58:102,118:162]                  # 하나를 견본으로
res=cv2.matchTemplate(tm, templ, cv2.TM_CCOEFF_NORMED)
ys,xs=np.where(res>=0.7)

# 가까운 검출을 하나로 묶어 개수 세기
found=[]
for x,y in sorted(zip(xs,ys)):
    if all(abs(x-fx)>20 for fx,_ in found): found.append((x,y))
print('찾은 마커 개수 :', len(found))

## 8. 꼭짓점 찾기 (approxPolyDP)
윤곽선을 다각형으로 근사해 모서리 좌표를 얻습니다.

In [ ]:
poly=np.zeros((200,300),np.uint8)
cv2.fillPoly(poly,[np.array([[60,40],[250,60],[230,170],[40,150]])],255)
cnts,_=cv2.findContours(poly, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
eps=0.02*cv2.arcLength(cnts[0], True)
corners=cv2.approxPolyDP(cnts[0], eps, True)
print('꼭짓점 개수 :', len(corners))
print('좌표 :', corners.reshape(-1,2).tolist())

---
예제 실행을 마친 후 `06_practice.ipynb` 의 문제를 해결하십시오.